# El sensor miente, y se puede medir cuánto

El AS5600 de este banco no mide el ángulo que uno cree. Su hoja de datos promete
del orden de medio grado con el imán bien puesto, y ese "bien puesto" tiene una
tolerancia de un cuarto de milímetro entre el eje de giro del imán y el centro del
integrado. Lo que sobra de esa tolerancia no aparece como ruido: aparece como una
función fija del ángulo que se repite vuelta tras vuelta, y a un lazo de control
se le presenta como una ondulación de velocidad que ninguna ganancia arregla.

Este notebook hace tres cosas, en este orden:

1. **Mide** el error, sin tener ningún encoder de referencia contra el cual
   comparar. La única regla disponible es el tiempo.
2. **Decide** cuánto de lo que midió es realmente el sensor y cuánto es el motor.
   Ésta es la parte difícil y es la que hace falta entender.
3. **Corrige**: arma una tabla, la mete en el Arduino, y vuelve a medir para ver
   si sirvió.

El método completo, con las cuentas, está en `Docs/CALIBRACION_AS5600.md`.

> **Para orientarse.** Poner `dev` en una celda muestra todo lo que la placa
> tiene: cada parámetro con su valor de ahora, su unidad, si se puede mover y una
> línea de qué es. `dev.describe('ang')` filtra por subsistema, y `dev.<TAB>`
> completa los nombres. No hay ninguna lista escrita de este lado: la placa
> declara la suya al conectarse.


In [ ]:
import sys, os
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import calib
from banco_simulado import conseguir_banco

# --- cómo se ven los gráficos de este notebook -------------------------------
# Cuatro colores, en orden fijo, elegidos para que se distingan también con
# daltonismo. La tinta del texto nunca lleva el color de una serie: el color es
# identidad de la marca, no del número.
AZUL, NARANJA, AQUA, AMARILLO = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
TINTA, TENUE, NUBE = '#0b0b0b', '#52514e', '#c9c8c3'

plt.rcParams.update({
    'figure.figsize': (9, 3.6), 'figure.dpi': 110,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#e6e5e1',
    'grid.linewidth': 0.8, 'axes.edgecolor': '#c9c8c3', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': TENUE, 'axes.titlecolor': TINTA, 'axes.titlelocation': 'left',
    'axes.titleweight': 'medium', 'axes.titlepad': 10,
    'xtick.color': TENUE, 'ytick.color': TENUE, 'text.color': TINTA,
    'lines.linewidth': 1.8, 'legend.frameon': False, 'font.size': 10,
})

# El banco de verdad si el cable está enchufado; si no, uno simulado que lo dice.
dev = conseguir_banco(forzar_simulado=os.environ.get('CALIB_SIMULADO') == '1')

## 1. Antes de medir nada: ¿está bien montado?

Ninguna tabla arregla un imán a la distancia equivocada. Si el control automático
de ganancia del AS5600 está contra un extremo, el sensor ya está trabajando fuera
de su rango y el error deja de ser una función suave del ángulo, que es lo único
que una tabla sabe corregir.

La línea que importa es la del imán: **AGC lejos de 0 y lejos de 255**. Si está
contra un borde, la respuesta correcta es un destornillador, no este notebook.

In [ ]:
dev.bringup()

## 2. El piso: ¿cuánto ruido tiene el sensor quieto?

Todo lo que sigue son mediciones de fracciones de grado, así que primero hay que
saber cuál es la fracción más chica que este banco puede distinguir. Con el eje
quieto, lo que se mueve es ruido.

Una cuenta del AS5600 son 0,088 grados. La hoja de datos promete 0,043 grados RMS
con el filtro rápido, que es media cuenta. **Nada que sea más chico que unas pocas
veces este número es una medición.**

In [ ]:
dev.rest()
df = dev.capture(5.0, warn=False)
_, quieto = calib.serie(df)

desvio = np.std(quieto)

plt.figure(figsize=(9, 3.0))
plt.hist(quieto - np.median(quieto), bins=np.arange(-4.25, 4.5, 0.5),
         color=AZUL, edgecolor='white', linewidth=1.2)
plt.axvline(0, color=TENUE, lw=1, ls='--')
plt.xlabel('desvío respecto de la mediana [cuentas]')
plt.ylabel('muestras')
plt.title(f'Piso de ruido con el eje quieto: {desvio:.2f} cuentas RMS '
          f'({desvio * calib.GRADOS_POR_CUENTA:.3f} grados)')
plt.show()

print(f'desvío estándar   {desvio:.2f} cuentas = {desvio*calib.GRADOS_POR_CUENTA:.3f} grados')
print(f'piso de detección {3*desvio:.2f} cuentas: por debajo de esto no hay medición')

## 3. Ver el problema

Ahora se hace girar el motor a comando constante y se mira el ángulo. La idea es
la única que tenemos: **si la velocidad es constante, el ángulo verdadero es una
recta en el tiempo**. Entonces se le resta al ángulo medido su propia marcha suave
y lo que queda tendría que ser cero.

No es cero. Y lo interesante es cómo no lo es: graficado contra el **ángulo dentro
de la vuelta** en lugar de contra el tiempo, cien vueltas se apilan una encima de
la otra y dibujan la misma curva. Eso es lo que quiere decir "el error es una
función del ángulo": no es ruido, es una firma, y se repite.

In [ ]:
df = calib.regimen(dev, uff=200, duracion=20.0)

t, cuentas = calib.serie(df)
ang, r = calib.residuo(t, cuentas)
centros, medio, n = calib.promediar(ang, r)

vueltas = np.ptp(cuentas) / calib.CUENTAS
print(f'{vueltas:.0f} vueltas en {np.ptp(t):.1f} s = {vueltas/np.ptp(t):.2f} rev/s, '
      f'{len(t)/vueltas:.0f} muestras por vuelta')

plt.scatter(ang, r, s=3, color=NUBE, edgecolors='none', label='cada muestra')
plt.plot(centros, medio, color=AZUL, lw=2, label='promedio de todas las vueltas')
plt.axhline(0, color=TENUE, lw=1, ls='--')
plt.xlim(0, calib.CUENTAS)
plt.xlabel('ángulo dentro de la vuelta [cuentas]')
plt.ylabel('error [cuentas]')
plt.title('El mismo error en cada vuelta, no ruido')
plt.legend(loc='upper right')
plt.show()

print(f'excursión del promedio: {np.nanmax(medio) - np.nanmin(medio):.2f} cuentas '
      f'= {(np.nanmax(medio) - np.nanmin(medio))*calib.GRADOS_POR_CUENTA:.2f} grados')

## 4. Ponerle números: los armónicos

Esa curva se describe con muy pocos números. Un imán descentrado da un error que
va **una vez por vuelta**; un imán inclinado da uno que va **dos veces por
vuelta**. Es la misma taxonomía que la de una figura de Lissajous: un corrimiento
de cero en los canales del sensor sale como primer armónico, una diferencia de
ganancia entre los canales sale como segundo.

Así que se ajusta

$$m(\theta) = \underbrace{c_0 + c_1 t + \dots}_{\text{la marcha del eje}} \;+\; \sum_{k=1}^{8} A_k \sin\!\left(\frac{2\pi k \theta}{4096} + \varphi_k\right)$$

**los dos términos a la vez, en un solo problema de mínimos cuadrados.** Por
separado no: la tendencia y el primer armónico compiten por la misma varianza, y
una tendencia demasiado flexible se come justamente lo que se quiere medir. Lo que
los mantiene separados es que viven en dominios distintos: la tendencia es suave
en el *tiempo*, el error es periódico en el *ángulo*.

In [ ]:
arm = calib.ajustar(t, cuentas)

# El residuo es el control de calidad del ajuste, y hay que mirarlo. Si da mucho
# más grande que el piso de ruido, el modelo no describió los datos: casi siempre
# es un transitorio de velocidad adentro de la ventana, y ahí las amplitudes que
# siguen no valen nada.
print(f'{arm.vueltas:.0f} vueltas a {arm.omega:.2f} rev/s, '
      f'residuo {arm.residuo:.2f} cuentas (el piso era {desvio:.2f})')
if arm.residuo > 3 * desvio:
    print('  OJO: el residuo no baja al piso, el ajuste no describe estos datos')
print()
for k in range(1, 5):
    print(f'  k={k}   A = {arm.A[k-1]:5.2f} +- {arm.sigma[k-1]:.2f} cuentas'
          f'   = {arm.A[k-1]*calib.GRADOS_POR_CUENTA:5.3f} grados'
          f'   fase {arm.phi[k-1]:+.2f} rad')

eje = np.arange(calib.CUENTAS)
plt.plot(centros, medio, color=AZUL, lw=2, label='medido')
plt.plot(eje, arm.evaluar(eje), color=NARANJA, lw=2, ls='--', label='ajuste, 8 armónicos')
plt.annotate('medido', (centros[6], medio[6]), textcoords='offset points',
             xytext=(6, 10), color=AZUL, fontsize=9)
plt.annotate('ajuste', (eje[2600], arm.evaluar(eje)[2600]), textcoords='offset points',
             xytext=(6, -16), color=NARANJA, fontsize=9)
plt.axhline(0, color=TENUE, lw=1, ls='--')
plt.xlim(0, calib.CUENTAS)
plt.xlabel('ángulo dentro de la vuelta [cuentas]')
plt.ylabel('error [cuentas]')
plt.title('Ocho números describen la curva entera')
plt.legend(loc='upper right')
plt.show()

## 5. La pregunta difícil: ¿es el sensor, o es el motor?

Acá es donde una calibración hecha de apuro se rompe.

Un motor de continua con escobillas **no gira a velocidad constante**. Tiene
ondulación de par por conmutación, tiene *cogging*, tiene rozamiento que depende
del ángulo. Y todo eso está enganchado al ángulo igual que el error del sensor. En
una sola corrida a una sola velocidad, las dos cosas son literalmente
indistinguibles: las dos producen un residuo periódico en el ángulo.

Si se calibra sin separarlas, la tabla "corrige" la dinámica del motor metiéndola
adentro del sensor, y el resultado es **peor que no hacer nada**.

Hay que saber cuál discriminador sirve:

| | ¿separa el sensor del motor? |
|---|---|
| Invertir el sentido de giro | **No.** El término de inercia es par en la velocidad, así que la ondulación mecánica tampoco cambia de signo. Sirve para otra cosa: cancela el retardo del filtro del sensor. |
| Barrer la velocidad | **Sí.** Y es el único. |

El error del sensor es una función del ángulo y no sabe a qué velocidad se lo
recorre: su amplitud es **constante** en $\omega$. Una ondulación de par tiene que
atravesar la mecánica para volverse posición, y eso la divide por la inercia:

$$|\delta_k| = \frac{T_k}{\sqrt{(Jk^2\omega^2)^2 + (bk\omega)^2}} \;\propto\; \omega^{-1} \dots \omega^{-2}$$

**En un gráfico log-log de amplitud contra velocidad, el sensor es una recta
horizontal y el motor baja.**

Para barrer la velocidad no hace falta hacer muchas corridas: alcanza con llevar
el motor a velocidad y soltarlo. Durante la desaceleración no hay corriente de
armadura, así que ni siquiera está la ondulación de conmutación, y la velocidad
barre sola de alta a baja.

In [ ]:
df_baja = calib.desaceleracion(dev, uff=220, duracion=25.0, arranque=3.0)

t2, c2 = calib.serie(df_baja)
plt.plot(t2, np.abs(np.gradient(c2, t2)) / calib.CUENTAS, color=AZUL)
plt.xlabel('t [s]'); plt.ylabel('velocidad [rev/s]')
plt.title('Una sola captura barre todo el rango de velocidad')
plt.show()

ventanas = calib.barrido(df_baja)
print(f'{len(ventanas)} ventanas, de {min(v.omega for v in ventanas):.1f} '
      f'a {max(v.omega for v in ventanas):.1f} rev/s')

In [ ]:
w = np.array([v.omega for v in ventanas])
orden = np.argsort(w)

plt.figure(figsize=(9, 4.0))
for k, color in zip((1, 2, 3, 4), (AZUL, NARANJA, AQUA, AMARILLO)):
    A = np.array([v.A[k-1] for v in ventanas])[orden]
    plt.plot(w[orden], A, 'o-', color=color, markersize=5)
    # Etiqueta directa sobre la línea: la identidad no queda librada al color.
    plt.annotate(f'k={k}', (w[orden][-1], A[-1]), textcoords='offset points',
                 xytext=(8, -3), color=color, fontsize=9, fontweight='medium')

plt.axhline(calib.A_MINIMA, color=TENUE, lw=1, ls=':')
plt.annotate('piso de aceptación', (w.min(), calib.A_MINIMA), textcoords='offset points',
             xytext=(2, 4), color=TENUE, fontsize=8)
plt.xscale('log'); plt.yscale('log')
plt.gca().xaxis.set_major_formatter(mticker.ScalarFormatter())
plt.gca().xaxis.set_minor_formatter(mticker.NullFormatter())
plt.gca().set_xticks([2, 3, 4, 6, 8, 12])
plt.xlabel('velocidad [rev/s]'); plt.ylabel('amplitud [cuentas]')
plt.title('Plano = el sensor.  Cae = el motor.')
plt.subplots_adjust(right=0.88)
plt.show()

for k in (1, 2, 3, 4):
    p = calib.pendiente(ventanas, k)
    print(f'  k={k}   pendiente {p:+.2f} en log-log   '
          f'-> {"el sensor" if abs(p) < 0.3 else "depende de la velocidad: el motor"}')

### La decisión

`calib.aceptar()` aplica la compuerta: un armónico entra en la tabla sólo si
supera el piso, supera su propia incertidumbre, y es plano en la velocidad. Lo que
no la pasa se descarta con el motivo escrito.

In [ ]:
aceptados, rechazos = calib.aceptar(ventanas)

## 6. La tabla

El error aceptado se muestrea en 64 ángulos y se guarda en **octavos de cuenta**,
que es un byte por entrada. 64 bytes en total.

Por qué octavos y no cuentas enteras: el error entero mide unas pocas cuentas, así
que en cuentas enteras la tabla tendría tres o cuatro valores distintos y sería un
escalón, no una corrección. El Arduino interpola linealmente entre entradas y
redondea al final.

**La tabla no se guarda en la placa.** El dispositivo arranca siempre sin
calibrar, y la dueña de la tabla es esta computadora, que la empuja al conectarse.
No es una limitación de memoria, es una decisión: una calibración es una propiedad
de *este banco* --este imán, en este eje-- y no del programa. En un archivo se lee,
se compara y se revisa; adentro de la placa es estado invisible que sobrevive a la
reprogramación. El caso feo no es la tabla que falta: es la tabla vieja, de otro
montaje, aplicándose en silencio.

In [ ]:
cal = calib.Calibracion.desde_armonicos(aceptados, banco='banco del aula')

eje = np.arange(calib.CUENTAS)
nodos = np.arange(0, calib.CUENTAS, calib.CUENTAS // calib.LUT_SIZE)
plt.plot(eje, aceptados.evaluar(eje), color=NUBE, lw=4, label='error modelado, continuo')
plt.step(eje, cal.corregir(eje), where='mid', color=NARANJA, lw=1.4,
         label='lo que hace la placa, en cuentas enteras')
plt.scatter(nodos, np.array(cal.lut) / 8.0, s=32, color=AZUL, zorder=3,
            label='las 64 entradas de la tabla')
plt.axhline(0, color=TENUE, lw=1, ls='--')
plt.xlim(0, calib.CUENTAS)
plt.xlabel('ángulo crudo [cuentas]'); plt.ylabel('corrección [cuentas]')
plt.title('Del modelo continuo a los 64 bytes que corren en el AVR')
plt.legend(loc='upper right')
plt.show()

print(f'tabla: {min(cal.lut)} a {max(cal.lut)} octavos de cuenta, '
      f'suma de verificación {cal.checksum():#06x}')

In [ ]:
# 64 escrituras de un parámetro común, y una sola lectura para verificarlas: el
# dispositivo mantiene la suma de Fletcher de lo que tiene, y acá se la compara
# contra la de lo que se quiso mandar. No hizo falta agregarle un comando al
# protocolo.
cal.aplicar(dev)
print(f'tabla puesta y correccion prendida: {calib.esta_puesta(dev, cal)}')

## 7. ¿Sirvió?

La única forma de saberlo es repetir la misma medición con la corrección apagada y
prendida, en la misma sesión y sin tocar nada más. Por eso `ang_cal` es un parámetro y
no una constante de compilación.

Ojo con qué canal se mira. `y_raw` es la cuenta cruda y **no cambia** al prender la
corrección: es justamente lo que la tabla no toca. Lo que hay que mirar es `y_uw`,
que es el ángulo que el lazo realmente usa.

In [ ]:
antes   = calib.regimen(dev, uff=200, duracion=15.0, cal=0)
despues = calib.regimen(dev, uff=200, duracion=15.0, cal=1)

a_antes   = calib.ajustar(*calib.serie(antes,   fuente='y_uw'))
a_despues = calib.ajustar(*calib.serie(despues, fuente='y_uw'))

# Primero la imagen que se entiende sola: el mismo gráfico del paso 3, con la
# corrección apagada y prendida.
for etiqueta, d, color in (('cal apagada', antes, AZUL),
                           ('cal prendida', despues, NARANJA)):
    ts, cs = calib.serie(d, fuente='y_uw')
    a, prom, _ = calib.promediar(*calib.residuo(ts, cs))
    plt.plot(a, prom * calib.GRADOS_POR_CUENTA, color=color, lw=2, label=etiqueta)

plt.axhline(0, color=TENUE, lw=1, ls='--')
plt.xlim(0, calib.CUENTAS)
plt.xlabel('ángulo dentro de la vuelta [cuentas]')
plt.ylabel('error de ángulo [grados]')
plt.title('El ángulo que usa el lazo, antes y después')
plt.legend(loc='upper right')
plt.show()

ks = np.arange(1, 5)
ancho = 0.38
plt.figure(figsize=(9, 3.4))
plt.bar(ks - ancho/2, a_antes.A[:4],   ancho, color=AZUL,
        edgecolor='white', linewidth=1.5, label='cal apagada')
plt.bar(ks + ancho/2, a_despues.A[:4], ancho, color=NARANJA,
        edgecolor='white', linewidth=1.5, label='cal prendida')
plt.xticks(ks, [f'k={k}' for k in ks])
plt.ylabel('amplitud [cuentas]')
plt.title('Lo que ve el lazo, antes y después de la corrección')
plt.legend(loc='upper right')
plt.show()

for k in ks:
    a, d = a_antes.A[k-1], a_despues.A[k-1]
    print(f'  k={k}   {a:5.2f} -> {d:5.2f} cuentas'
          + (f'   ({a/d:.1f} veces menos)' if d > 0.05 else ''))

### La compuerta

El criterio de éxito estaba escrito antes de mirar el resultado: el primer y el
segundo armónico tienen que caer **a menos de un quinto**. Si no bajan, la tabla
está mal indexada o tiene el signo dado vuelta. Si bajan pero la velocidad no se
alisa, lo que quedaba era mecánico y no había nada que corregir.

In [ ]:
for k in (1, 2):
    a, d = a_antes.A[k-1], a_despues.A[k-1]
    if a < calib.A_MINIMA:
        print(f'  k={k}: no había nada que corregir ({a:.2f} cuentas)')
    else:
        print(f'  k={k}: {"PASA " if d < a/5 else "NO PASA"}  '
              f'{a:.2f} -> {d:.2f} cuentas')

s_antes   = np.std(np.gradient(calib.serie(antes,   fuente='y_uw')[1]))
s_despues = np.std(np.gradient(calib.serie(despues, fuente='y_uw')[1]))
print(f'\nondulación de velocidad: {s_antes:.2f} -> {s_despues:.2f} cuentas por muestra')

## 8. Y lo que importa de verdad: una velocidad que no existe

Todo lo anterior mira el sensor. Lo que le importa a alguien que no esté mirando
el sensor es qué le hace esto a su lazo, y la respuesta es concreta: **el error de
ángulo se deriva en una ondulación de velocidad que el eje no tiene.**

Si el ángulo medido es $\theta + e(\theta)$, la velocidad medida es
$\dot\theta\,(1 + e'(\theta))$. Con seis cuentas de primer armónico eso es cerca
de un uno por ciento de ondulación, a una vez por vuelta, sobre un eje que gira
parejo. Cualquier cosa que consuma velocidad --un lazo de velocidad, el término
derivativo de un PID-- la persigue, y al perseguirla la vuelve real.

Dos advertencias, que son la parte útil de esta sección.

**El lugar donde esto NO se ve es un escalón de posición.** Un motor con escobillas
contra un puente Darlington tiene una zona muerta de fricción de varias cuentas,
así que el eje queda parado donde se traba, y esa banda es más grande que el error
del sensor. La corrección está ahí igual; lo que la tapa es la mecánica.

**Y en la velocidad no queda cero, queda el motor.** La derivada multiplica cada
armónico por su orden, así que la ondulación de par que se descartó en el paso 5
--con razón, no es del sensor-- pesa más acá que en el ángulo. Que sobreviva es la
prueba de que no se la calibró.

In [ ]:
for etiqueta, d in (('cal apagada', antes), ('cal prendida', despues)):
    ts, cs = calib.serie(d, fuente='y_uw')
    v = np.gradient(cs, ts) / calib.CUENTAS                  # rev/s
    _, prom, _ = calib.promediar(np.mod(cs, calib.CUENTAS), v)
    print(f'  {etiqueta:<14} {np.nanmax(prom) - np.nanmin(prom):.3f} rev/s pico a pico '
          f'sobre {np.nanmean(prom):.2f} rev/s')

print('\nLo que queda es el motor, y está bien que quede: nunca entró en la tabla.')

## 9. Guardar la calibración

El archivo JSON es la calibración. Lleva la tabla, de dónde salió y cuándo, y su
propia suma de verificación: un archivo editado a mano no se aplica en silencio,
protesta.

Al principio de cualquier otro notebook, `calib.asegurar(dev)` lo carga y lo
empuja a la placa si no está ya puesto.

Para un tablero que se enciende solo y nadie conecta a una computadora,
`escribir_header()` genera `Banco/Calibracion.h` y el sketch lo toma en la
próxima compilación. Ése es el único camino en el que la calibración queda
adentro de la placa, y queda a la vista en el código en lugar de escondida.

Pero tiene un filo, y por eso está apagado: una vez escrito, el header entra en
**toda** compilación siguiente y el sketch arranca con `cal = 1`. Escribir uno
medido acá y después ir al banco de verdad es la manera exacta de calibrar con la
tabla equivocada sin enterarse. Prender `DEJAR_COMPILADA` a sabiendas, y borrar el
archivo cuando deje de corresponder.

In [ ]:
cal.notas = f'{aceptados.vueltas:.0f} vueltas, residuo {aceptados.residuo:.2f} cuentas'

# Un banco de mentira no escribe el archivo del banco de verdad. Parece una
# obviedad y no lo es: la primera vez que se corrio esto en modo demostracion, la
# calibracion simulada piso la medida en el banco, y el notebook de control cargo
# la equivocada sin que nadie se enterara.
RUTA = 'calibracion-simulada.json' if getattr(dev, 'simulado', False) else 'calibracion.json'
print(cal.guardar(RUTA))

# Y el header, sólo si alguien lo pide a propósito. Escribirlo NO es inocuo:
# el sketch lo toma en la compilación siguiente y arranca con `cal = 1`, así que
# a partir de ahí toda grabación lleva esta tabla adentro aunque nadie se acuerde.
# Escribir uno medido en el banco simulado y después ir al banco de verdad es la
# forma exacta de calibrar con la tabla equivocada sin enterarse. Pasó.
DEJAR_COMPILADA = False

if DEJAR_COMPILADA:
    print(cal.escribir_header('../Banco/Calibracion.h'))
else:
    print('el header no se escribió; poner DEJAR_COMPILADA = True para dejarla en el sketch')

## Si algo no cerró

| Lo que se ve | Lo que suele ser |
|---|---|
| El AGC contra un borde en el paso 1 | El imán está a la distancia equivocada. Destornillador, no tabla. |
| El error no se repite entre vueltas | El imán está flojo en el eje. |
| Todos los armónicos caen con la velocidad | No hay error de sensor medible: lo que se veía era el motor. Es un resultado. |
| Al prender `ang_cal` el error se duplica | Signo invertido: la tabla se está sumando en lugar de restarse. |
| Los armónicos altos aparecen y desaparecen | Pocas muestras por vuelta. Bajar `loop_div` o frenar el motor. |
| Las amplitudes cambian entre corridas iguales | Ventanas de ajuste demasiado cortas: menos de 20 vueltas y la tendencia se come el primer armónico. |